# Prototype: response distributions on EGMA math (multiple choice)

This notebook documents a **small-scale prototype** for studying how vision–language models behave on **LEVANTE EGMA math** items when we (1) draw **multiple stochastic samples** under a fixed prompt and (2) vary **option order** via controlled shuffle seeds (a **robustness** axis).

**Branch:** `prototype/math-response-distribution`  
**Code:** `levante_bench.evaluation.stochastic`, extended `generate()` sampling kwargs on HF-backed models, and `egma_math` helpers to rebuild trials with alternate shuffle RNGs.

## 1. Experimental design

### 1.1 Task and models

- **Task:** `egma-math` — text-only multiple choice (letters `A`–`D`, etc.). Trials are built from the public corpus CSV; audio-dependent rows are skipped (same rule as `EgmaMathDataset`).
- **Models (example pair):** `SmolVLM2` and `Qwen3.5-VL` configs from `configs/models/` (`smolvlm2`, `qwen35`). Both support `text_only` items; the runner-style prompt adds the JSON answer instruction (`evaluate_trial` parity is preserved via `eval_prompt_for_trial()` in code).

### 1.2 Axis A — Distribution over answer letters (fixed presentation)

- **Procedure:** Fix one corpus row and a **fixed shuffle seed** so the option order (and thus the full prompt) is constant. Run **K independent stochastic decodes** with `do_sample=True`, `temperature`, `top_p`, and distinct `sample_seed` values.
- **Output:** Empirical histogram over parsed labels `{A,B,C,D,…}` plus an `_unparsed` bucket when JSON/letter parsing fails.
- **Interpretation:** This is **not** a calibrated probability over options; it is the **sampling-induced spread** of the model’s *policy* under the chosen decoding hyperparameters. High entropy suggests inconsistent verbalization of the same underlying item; a spike on one letter suggests a nearly deterministic preference *given that surface form*.

### 1.3 Axis B — Shuffle robustness (varying presentation)

- **Procedure:** Keep the **same underlying item** (same `item_uid` / answer text) but vary an integer **shuffle seed** passed into `math_shuffle_rng(row, shuffle_seed)`. Each seed induces a (usually) different permutation of options while preserving the correct answer **content**.
- **Decoding:** One **greedy** pass per shuffle (`do_sample=False`) keeps compute modest and isolates **layout sensitivity** from sampling noise.
- **Metrics:** (i) **accuracy vs. shuffle seed**; (ii) **chosen option text** (not raw letter) to avoid mixing semantics when comparing across seeds.
- **Interpretation:** Strong dependence on shuffle seed indicates **position / ordering bias** or fragility to surface form, even when the mathematical content is unchanged.

### 1.4 Practical limits

- GPU memory and Hugging Face download time dominate; reduce `K` or the number of shuffle seeds on smaller machines.
- Stochastic behaviour can still depend on **non-deterministic CUDA kernels** unless you enable strict determinism (not required for this exploratory notebook).

## 2. Setup

Run from the **repository root** with the package importable (`pip install -e .` in a venv, or extend `sys.path` with `src/` as below). You also need **`data/assets/<version>/corpus/egma-math/...`** (see project README / `scripts/download_levante_assets.py`).

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from omegaconf import OmegaConf

REPO_ROOT = Path.cwd().resolve()
SRC = REPO_ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from levante_bench.config.defaults import detect_data_version
from levante_bench.config.loader import load_model_config
from levante_bench.evaluation.runner import resolve_device
from levante_bench.evaluation.stochastic import (
    collect_parsed_labels,
    greedy_predicted_label,
    label_histogram,
)
from levante_bench.models import get_model_class
from levante_bench.tasks.egma_math import (
    egma_math_corpus_path,
    egma_math_trial_from_row,
    iter_egma_math_corpus_rows,
    math_shuffle_rng,
)

DATA_ROOT = Path(os.environ.get("LEVANTE_DATA_ROOT", REPO_ROOT / "data"))
VERSION = os.environ.get("LEVANTE_DATA_VERSION", "").strip() or detect_data_version(DATA_ROOT)

# Stochastic axis
K_SAMPLES = int(os.environ.get("MATH_DIST_K", "24"))
TEMPERATURE = float(os.environ.get("MATH_DIST_TEMP", "0.8"))
TOP_P = float(os.environ.get("MATH_DIST_TOP_P", "0.95"))
FIXED_SHUFFLE_SEED = int(os.environ.get("MATH_DIST_FIXED_SHUFFLE", "0"))

# Robustness axis: integer seeds passed to math_shuffle_rng
SHUFFLE_SEEDS = list(range(int(os.environ.get("MATH_DIST_SHUFFLE_MIN", "0")), int(os.environ.get("MATH_DIST_SHUFFLE_MAX", "12"))))

MODEL_IDS = ["smolvlm2", "qwen35"]

print("DATA_ROOT:", DATA_ROOT)
print("VERSION:", VERSION)
print("K_SAMPLES:", K_SAMPLES, "TEMPERATURE:", TEMPERATURE, "TOP_P:", TOP_P)

## 3. Select one math item (corpus row)

We pick the **first** eligible row that yields **at least four** options after filtering, so letter histograms are naturally four-way.

In [ ]:
corpus_path = egma_math_corpus_path(DATA_ROOT, VERSION)
if not corpus_path.exists():
    raise FileNotFoundError(
        f"Missing corpus: {corpus_path}\n"
        "Download assets (see README) or set LEVANTE_DATA_ROOT / LEVANTE_DATA_VERSION."
    )

selected_row = None
for row in iter_egma_math_corpus_rows(corpus_path):
    rng_probe = math_shuffle_rng(row, FIXED_SHUFFLE_SEED)
    trial_probe = egma_math_trial_from_row(row, rng_probe)
    if trial_probe is not None and len(trial_probe.get("option_labels", [])) >= 4:
        selected_row = row
        break

if selected_row is None:
    raise RuntimeError("No eligible 4-option item found in corpus (check CSV).")

ITEM_UID = (selected_row.get("item_uid") or "").strip()
print("item_uid:", ITEM_UID)

trial_fixed = egma_math_trial_from_row(
    selected_row, math_shuffle_rng(selected_row, FIXED_SHUFFLE_SEED)
)
assert trial_fixed is not None
trial_fixed["max_new_tokens"] = 128
print("gold:", trial_fixed["correct_label"])
print("labels:", trial_fixed["option_labels"])
print("--- prompt (truncated) ---")
print(trial_fixed["prompt"][:1200], "..." if len(trial_fixed["prompt"]) > 1200 else "")

## 4. Load two VLMs

Weights download from Hugging Face on first run. `resolve_device` follows the same `auto` behaviour as the evaluation runner.

In [ ]:
device = resolve_device(os.environ.get("LEVANTE_DEVICE", "auto"))
print("device:", device)


def load_model(model_id: str):
    cfg_dc = load_model_config(model_id)
    if cfg_dc is None:
        raise ValueError(f"No config for {model_id}")
    cfg = OmegaConf.to_container(cfg_dc, resolve=True)
    reg_name = cfg.get("name", model_id)
    cls = get_model_class(reg_name)
    if cls is None:
        raise ValueError(f"Model class not registered: {reg_name}")
    kwargs = dict(model_name=cfg["hf_name"], device=device)
    if "dtype" in cfg:
        kwargs["dtype"] = cfg["dtype"]
    if "attn_implementation" in cfg:
        kwargs["attn_implementation"] = cfg["attn_implementation"]
    m = cls(**kwargs)
    m.load()
    return m, cfg


loaded = {mid: load_model(mid) for mid in MODEL_IDS}

## 5. Axis A — Stochastic histograms (fixed prompt)

For each model we draw **K** samples and count parsed letters. Bars sum to K (including `_unparsed`).

In [ ]:
def plot_label_histogram(title: str, hist, order: list[str]):
    labels = [x for x in order if hist.get(x, 0) > 0]
    extras = sorted(k for k in hist if k not in order and hist[k] > 0)
    keys = labels + extras
    vals = [hist[k] for k in keys]
    colors = ["#4C72B0" if k != "_unparsed" else "#C44E52" for k in keys]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(keys, vals, color=colors)
    ax.set_ylabel("Count")
    ax.set_xlabel("Parsed label")
    ax.set_title(title)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.05 * max(vals or [1]), str(v), ha="center", fontsize=9)
    plt.tight_layout()
    return fig


letter_order = list(trial_fixed["option_labels"]) + ["_unparsed"]
stochastic_results = {}
for mid, (model, cfg) in loaded.items():
    mtok = int(cfg.get("max_new_tokens", trial_fixed.get("max_new_tokens", 128)))
    labs = collect_parsed_labels(
        model,
        trial_fixed,
        n_samples=K_SAMPLES,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        base_seed=0,
        max_new_tokens=mtok,
    )
    h = label_histogram(labs)
    stochastic_results[mid] = {"labels": labs, "hist": h}
    fig = plot_label_histogram(
        f"{mid}: stochastic labels (K={K_SAMPLES}, T={TEMPERATURE}, top_p={TOP_P})",
        h,
        letter_order,
    )
    plt.show()

for mid, pack in stochastic_results.items():
    h = pack["hist"]
    total = sum(h.values())
    mode = max(h, key=h.get) if h else None
    print(mid, "mode:", mode, "freq:", h.get(mode, 0) / total if total else 0.0)

## 6. Axis B — Shuffle robustness (greedy, one pass per seed)

We report **whether** the greedy choice matches the gold letter for that shuffle, and show **which option text** was chosen (stable across relabeling).

In [ ]:
def option_text_for_label(trial: dict, label: str | None) -> str:
    if not label:
        return "(unparsed)"
    labels = trial["option_labels"]
    opts = trial["options"]
    lu = label.upper()
    for i, lb in enumerate(labels):
        if str(lb).upper() == lu:
            return str(opts[i])[:80]
    return "(unknown label)"


robustness_rows = []
for mid, (model, cfg) in loaded.items():
    mtok = int(cfg.get("max_new_tokens", trial_fixed.get("max_new_tokens", 128)))
    for s in SHUFFLE_SEEDS:
        trial_s = egma_math_trial_from_row(selected_row, math_shuffle_rng(selected_row, s))
        if trial_s is None:
            continue
        trial_s["max_new_tokens"] = mtok
        pred = greedy_predicted_label(model, trial_s, max_new_tokens=mtok)
        gold = trial_s["correct_label"]
        robustness_rows.append(
            {
                "model": mid,
                "shuffle_seed": s,
                "gold": gold,
                "predicted": pred,
                "correct": pred == gold,
                "chosen_option": option_text_for_label(trial_s, pred),
            }
        )

def _print_table(rows: list[dict]) -> None:
    if not rows:
        print("(no rows)")
        return
    keys = list(rows[0].keys())
    print(" | ".join(keys))
    print("-" * (sum(len(k) for k in keys) + 3 * len(keys)))
    for r in rows:
        print(" | ".join(str(r[k]) for k in keys))


_print_table(robustness_rows)

for mid in MODEL_IDS:
    sub = [r for r in robustness_rows if r["model"] == mid]
    n = len(sub)
    acc = sum(1 for r in sub if r["correct"]) / n if n else float("nan")
    print(f"{mid} greedy accuracy over shuffle seeds: {acc:.3f} ({sum(1 for r in sub if r['correct'])}/{n})")

fig, axes = plt.subplots(1, len(MODEL_IDS), figsize=(4 * len(MODEL_IDS), 3), squeeze=False)
for ax, mid in zip(axes[0], MODEL_IDS):
    sub = [r for r in robustness_rows if r["model"] == mid]
    xs = [r["shuffle_seed"] for r in sub]
    ys = [int(r["correct"]) for r in sub]
    ax.scatter(xs, ys, alpha=0.7)
    ax.set_title(f"{mid}: correct vs shuffle seed")
    ax.set_xlabel("shuffle_seed")
    ax.set_ylabel("correct (0/1)")
    ax.set_yticks([0, 1])
plt.tight_layout()
plt.show()

## 7. Discussion and conclusions

1. **Stochastic axis (fixed prompt):** The histogram summarizes **how often** each letter is produced under repeated sampling. A narrow peak means the model is **stable** on that surface form; a flat or multi-modal histogram suggests **high generative variance** (or parsing failures landing in `_unparsed`). This does **not** replace item-level accuracy from a single greedy benchmark run — it characterises **within-item dispersion**.

2. **Shuffle axis (robustness):** If accuracy **flips** across seeds while the mathematical content is fixed, the model is sensitive to **option ordering / framing**. Comparing **chosen option text** (not the letter) keeps the readout aligned with **semantic choice** when labels permute.

3. **Next steps (if promoting beyond prototype):** wire sampling kwargs into the main `runner` behind a config flag; cache keys should include `(do_sample, temperature, top_p, sample_seed)`; add optional **batched** generation for throughput; document **determinism** expectations per backend (CUDA, MPS, CPU).

4. **Limitations:** Two small VLMs on **one** item are illustrative only. Temperature/top_p choices shift the histogram; shuffle seed range and item choice should be expanded for any quantitative claim.